# ScamShield Z
**Real-Time, Privacy-Preserving Detection of Payment Scams and Mule Accounts on IBM LinuxONE**

IBM Z Datathon 2026 | Challenge: Real-Time AI for Critical Decisions | Team: Operation Code

### Problem
Authorised push-payment scams trick victims into approving payments. Money moves through mule accounts in minutes. Rule-based checks miss these because the payment looks legitimate.

### Solution
Score every payment **before** it completes → **ALLOW / WARN / HOLD** with human-readable reasons (EN / HI / TE), plus a **bank-manager investigation queue**.

### Stack
PaySim 2M sample → behaviour + mule-style network features → XGBoost → optional ONNX → IBM LinuxONE


## 1. Setup


In [1]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_fscore_support,
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import onnxmltools
    from onnxmltools.convert.common.data_types import FloatTensorType
    HAS_ONNX = True
except Exception:
    HAS_ONNX = False

print("ScamShield Z setup ready on IBM LinuxONE")
print("XGBoost available:", HAS_XGBOOST)
print("ONNX export tools:", HAS_ONNX)


ScamShield Z setup ready on IBM LinuxONE
XGBoost available: True
ONNX export tools: False


## 2. Data — PaySim (2M sample)
Primary dataset from abstract. Account IDs used for graph features only (privacy-preserving story).


In [2]:
candidates = [Path("paysim_2m.csv"), Path("shared/paysim_2m.csv")]
data_path = next((p for p in candidates if p.exists()), None)
assert data_path is not None, "Upload paysim_2m.csv to notebook folder or shared/"

df = pd.read_csv(data_path)
print("loaded:", data_path, "shape:", df.shape)
print(df["isFraud"].value_counts())
df.head(3)


loaded: paysim_2m.csv shape: (2000000, 11)
isFraud
0    1997452
1       2548
Name: count, dtype: int64


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,278,CASH_IN,330218.42,C632336343,20866.0,351084.42,C834976624,452419.57,122201.15,0,0
1,15,PAYMENT,11647.08,C1264712553,30370.0,18722.92,M215391829,0.00,0.00,0,0
2,10,CASH_IN,152264.21,C1746846248,106589.0,258853.21,C1607284477,201303.01,49038.80,0,0


## 3. Features — behaviour + mule-style network signals
Balance/behaviour signals + fan-in (many senders to same receiver).


In [3]:
df_model = df.copy()

df_model["errorBalanceOrig"] = (
    df_model["oldbalanceOrg"] - df_model["amount"] - df_model["newbalanceOrig"]
)
df_model["errorBalanceDest"] = (
    df_model["oldbalanceDest"] + df_model["amount"] - df_model["newbalanceDest"]
)
df_model["orig_zero"] = (
    (df_model["oldbalanceOrg"] == 0) & (df_model["newbalanceOrig"] == 0)
).astype(int)
df_model["dest_zero"] = (
    (df_model["oldbalanceDest"] == 0) & (df_model["newbalanceDest"] == 0)
).astype(int)
df_model["amount_log"] = np.log1p(df_model["amount"])
df_model["hour_proxy"] = df_model["step"] % 24

recv_stats = df_model.groupby("nameDest").agg(
    dest_txn_count=("nameOrig", "size"),
    dest_unique_senders=("nameOrig", "nunique"),
    dest_total_amount=("amount", "sum"),
)
df_model = df_model.merge(recv_stats, left_on="nameDest", right_index=True, how="left")
df_model["dest_fan_in"] = df_model["dest_unique_senders"]
df_model["receiver_is_merchant"] = df_model["nameDest"].astype(str).str.startswith("M").astype(int)

le = LabelEncoder()
df_model["type_enc"] = le.fit_transform(df_model["type"])

features = [
    "step", "type_enc", "amount", "amount_log", "hour_proxy",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "errorBalanceOrig", "errorBalanceDest",
    "orig_zero", "dest_zero",
    "dest_txn_count", "dest_unique_senders", "dest_fan_in",
    "dest_total_amount", "receiver_is_merchant",
]

X = df_model[features]
y = df_model["isFraud"]
print("feature count:", len(features))
print(df_model["dest_fan_in"].describe())


feature count: 18
count    2.000000e+06
mean     4.201089e+00
std      4.158872e+00
min      1.000000e+00
25%      1.000000e+00
50%      3.000000e+00
75%      6.000000e+00
max      4.100000e+01
Name: dest_fan_in, dtype: float64


## 4. Split — 70% train / 15% val / 15% test (stratified)


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print(
    "fraud -> train:", int(y_train.sum()),
    "val:", int(y_val.sum()),
    "test:", int(y_test.sum()),
)


train: (1400000, 18) val: (300000, 18) test: (300000, 18)
fraud -> train: 1784 val: 382 test: 382


## 5. Train XGBoost (class-weighted for imbalance)


In [5]:
assert HAS_XGBOOST, "Install/enable XGBoost on this kernel"

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
    random_state=42, n_jobs=-1, early_stopping_rounds=30,
)
t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
train_secs = time.time() - t0

train_proba = model.predict_proba(X_train)[:, 1]
val_proba = model.predict_proba(X_val)[:, 1]
test_proba = model.predict_proba(X_test)[:, 1]

print("train_seconds:", round(train_secs, 2))
print("best_iteration:", model.best_iteration)
print("TRAIN ROC-AUC:", round(roc_auc_score(y_train, train_proba), 4),
      "PR-AUC:", round(average_precision_score(y_train, train_proba), 4))
print("VAL   ROC-AUC:", round(roc_auc_score(y_val, val_proba), 4),
      "PR-AUC:", round(average_precision_score(y_val, val_proba), 4))
print("TEST  ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4),
      "PR-AUC:", round(average_precision_score(y_test, test_proba), 4))
print("Note: PaySim is synthetic; high scores expected. Value is Allow/Warn/Hold + operator queue.")


train_seconds: 12.74
best_iteration: 147
TRAIN ROC-AUC: 1.0 PR-AUC: 1.0
VAL   ROC-AUC: 0.9994 PR-AUC: 0.9963
TEST  ROC-AUC: 1.0 PR-AUC: 1.0
Note: PaySim is synthetic; high scores expected. Value is Allow/Warn/Hold + operator queue.


## 6. Decisions — ALLOW / WARN / HOLD
- **ALLOW**: proceed, no warning text
- **WARN**: sender must confirm (careful multilingual message)
- **HOLD**: block/delay → bank analyst queue


In [6]:
HOLD_TH = 0.70
WARN_TH = 0.30

def decide(proba):
    if proba >= HOLD_TH:
        return "HOLD"
    if proba >= WARN_TH:
        return "WARN"
    return "ALLOW"

decisions = [decide(p) for p in test_proba]
y_pred_flag = np.array([0 if d == "ALLOW" else 1 for d in decisions])
print("Decision counts on TEST:")
print(pd.Series(decisions).value_counts())
print("\nConfusion matrix (true fraud vs flagged HOLD/WARN):")
print(confusion_matrix(y_test, y_pred_flag))
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred_flag, average="binary", zero_division=0
)
print(f"\nPrecision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")


Decision counts on TEST:
ALLOW    299614
HOLD        383
WARN          3
Name: count, dtype: int64

Confusion matrix (true fraud vs flagged HOLD/WARN):
[[299614      4]
 [     0    382]]

Precision: 0.9896  Recall: 1.0000  F1: 0.9948


## 7. Multilingual messages (EN / HI / TE)
- **ALLOW**: no warning text
- **WARN (sender)**: careful full reasons so the sender can decide
- **HOLD (sender)**: short block notice + top 1–2 reasons
- **HOLD (bank operator)**: clear action + reasons (also in EN / HI / TE)


In [7]:
WARN_CATALOG = {
    "receiver_zero_balance_history": {
        "en": "The receiving account shows little/no balance history (possible new or emptied mule account). Confirm the recipient before paying.",
        "hi": "प्राप्तकर्ता खाते में बैलेंस इतिहास कम/नहीं है (नया या खाली म्यूल अकाउंट हो सकता है)। भुगतान से पहले पुष्टि करें।",
        "te": "స్వీకర్త ఖాతాలో బ్యాలెన్స్ చరిత్ర తక్కువ/లేదు (కొత్త లేదా ఖాళీ మ్యూల్ ఖాతా కావచ్చు). చెల్లించే ముందు నిర్ధారించండి.",
    },
    "receiver_had_zero_before_credit": {
        "en": "Receiver balance was zero before this credit — a common scam pattern. Double-check who you are paying.",
        "hi": "इस क्रेडिट से पहले प्राप्तकर्ता बैलेंस शून्य था — आम स्कैम पैटर्न। पुष्टि करें कि आप किसे भुगतान कर रहे हैं।",
        "te": "ఈ క్రెడిట్‌కు ముందు స్వీకర్త బ్యాలెన్స్ సున్నా — సాధారణ స్కామ్ నమూనా. ఎవరికి చెల్లిస్తున్నారో మళ్లీ తనిఖీ చేయండి.",
    },
    "receiver_balance_inconsistency": {
        "en": "Receiver balance change does not match this payment cleanly. Pause and verify the destination.",
        "hi": "प्राप्तकर्ता बैलेंस परिवर्तन इस भुगतान से मेल नहीं खाता। रुकें और गंतव्य सत्यापित करें।",
        "te": "స్వీకర్త బ్యాలెన్స్ మార్పు ఈ చెల్లింపుతో సరిగా సరిపోలడం లేదు. ఆపి గమ్యాన్ని ధృవీకరించండి.",
    },
    "large_transfer_to_receiver": {
        "en": "Large amount to this receiver. If you were urged to pay quickly, stop and confirm out-of-band.",
        "hi": "इस प्राप्तकर्ता को बड़ी राशि। जल्दी भुगतान का दबाव हो तो रुकें और अन्य माध्यम से पुष्टि करें।",
        "te": "ఈ స్వీకర్తకు పెద్ద మొత్తం. త్వరగా చెల్లించమని ఒత్తిడి ఉంటే ఆపి వేరే మార్గంలో నిర్ధారించండి.",
    },
    "high_fan_in_mule": {
        "en": "This receiver is getting money from many different senders (mule-like fan-in). Do you personally know this person?",
        "hi": "यह प्राप्तकर्ता कई अलग-अलग भेजने वालों से पैसे ले रहा है (म्यूल जैसा पैटर्न)। क्या आप इस व्यक्ति को व्यक्तिगत रूप से जानते हैं?",
        "te": "ఈ స్వీకర్త అనేక మంది పంపినవారి నుండి డబ్బు అందుకుంటున్నారు (మ్యూల్ లాంటి నమూనా). మీకు ఈ వ్యక్తి వ్యక్తిగతంగా తెలుసా?",
    },
    "risky_type_TRANSFER": {
        "en": "TRANSFER to this receiver has elevated scam risk. Verify recipient before releasing funds.",
        "hi": "इस प्राप्तकर्ता को TRANSFER में ऊँचा स्कैम जोखिम है। राशि भेजने से पहले सत्यापित करें।",
        "te": "ఈ స్వీకర్తకు TRANSFER‌లో ఎక్కువ స్కామ్ ప్రమాదం ఉంది. నిధులు విడుదల చేసే ముందు ధృవీకరించండి.",
    },
    "risky_type_CASH_OUT": {
        "en": "CASH_OUT-linked pattern detected — often used after scam transfers. Verify carefully.",
        "hi": "CASH_OUT पैटर्न मिला — अक्सर स्कैम ट्रांसफर के बाद। सावधानी से सत्यापित करें।",
        "te": "CASH_OUT నమూనా కనిపించింది — తరచుగా స్కామ్ బదిలీల తర్వాత. జాగ్రత్తగా ధృవీకరించండి.",
    },
    "generic_medium_risk": {
        "en": "Medium scam risk. Extra verification (OTP / callback / payee confirmation) recommended before you approve.",
        "hi": "मध्यम स्कैम जोखिम। स्वीकृति से पहले अतिरिक्त सत्यापन (OTP / कॉलबैक) करें।",
        "te": "మధ్యస్థ స్కామ్ ప్రమాదం. ఆమోదించే ముందు అదనపు ధృవీకరణ (OTP / కాల్‌బ్యాక్) చేయండి.",
    },
}


def receiving_flags(row):
    flags = []
    try:
        txn_type = le.inverse_transform([int(row["type_enc"])])[0]
    except Exception:
        txn_type = "UNKNOWN"

    if int(row.get("dest_zero", 0)) == 1:
        flags.append("receiver_zero_balance_history")
    if float(row["oldbalanceDest"]) == 0 and float(row["amount"]) > 0:
        flags.append("receiver_had_zero_before_credit")
    if abs(float(row["errorBalanceDest"])) > 1:
        flags.append("receiver_balance_inconsistency")
    if float(row["amount"]) >= 100000:
        flags.append("large_transfer_to_receiver")
    if float(row.get("dest_fan_in", 0)) >= 20:
        flags.append("high_fan_in_mule")
    if txn_type == "TRANSFER":
        flags.append("risky_type_TRANSFER")
    if txn_type == "CASH_OUT":
        flags.append("risky_type_CASH_OUT")
    return flags, txn_type


HOLD_REASON_TEXT = {
    "high_fan_in_mule": {
        "en": "mule-like many-sender pattern on receiver",
        "hi": "प्राप्तकर्ता पर कई भेजने वालों जैसा म्यूल पैटर्न",
        "te": "స్వీకర్తపై ఎక్కువ మంది పంపినవారి మ్యూల్ నమూనా",
    },
    "receiver_zero_balance_history": {
        "en": "weak/empty receiver balance history",
        "hi": "प्राप्तकर्ता बैलेंस इतिहास कम/खाली",
        "te": "స్వీకర్త బ్యాలెన్స్ చరిత్ర బలహీనం/ఖాళీ",
    },
    "receiver_had_zero_before_credit": {
        "en": "receiver balance was zero before this credit",
        "hi": "इस क्रेडिट से पहले प्राप्तकर्ता बैलेंस शून्य था",
        "te": "ఈ క్రెడిట్‌కు ముందు స్వీకర్త బ్యాలెన్స్ సున్నా",
    },
    "large_transfer_to_receiver": {
        "en": "large transfer amount",
        "hi": "बड़ी राशि का ट्रांसफर",
        "te": "పెద్ద మొత్తం బదిలీ",
    },
    "risky_type_TRANSFER": {
        "en": "high-risk type (TRANSFER)",
        "hi": "उच्च जोखिम प्रकार (TRANSFER)",
        "te": "అధిక ప్రమాద రకం (TRANSFER)",
    },
    "risky_type_CASH_OUT": {
        "en": "high-risk type (CASH_OUT)",
        "hi": "उच्च जोखिम प्रकार (CASH_OUT)",
        "te": "అధిక ప్రమాద రకం (CASH_OUT)",
    },
    "generic": {
        "en": "elevated combined scam risk",
        "hi": "संयुक्त स्कैम जोखिम ऊँचा",
        "te": "కలిపి స్కామ్ ప్రమాదం ఎక్కువ",
    },
}


def hold_reason_keys(row):
    flags, txn_type = receiving_flags(row)
    keys = []
    for k in [
        "high_fan_in_mule",
        "receiver_zero_balance_history",
        "receiver_had_zero_before_credit",
        "large_transfer_to_receiver",
        "risky_type_TRANSFER",
        "risky_type_CASH_OUT",
    ]:
        if k in flags:
            keys.append(k)
    return (keys[:2] if keys else ["generic"])


def hold_message_sender(row, proba, lang="en"):
    keys = hold_reason_keys(row)
    why = "; ".join(HOLD_REASON_TEXT[k][lang] for k in keys)
    if lang == "hi":
        return (
            f"HOLD: भुगतान समीक्षा के लिए रोका/विलंबित (स्कैम जोखिम {proba:.1%})। "
            f"मुख्य संकेत: {why}। बैंक विश्लेषक जाँच तक राशि नहीं जाएगी।"
        )
    if lang == "te":
        return (
            f"HOLD: చెల్లింపు సమీక్ష కోసం ఆపబడింది/ఆలస్యం (స్కామ్ ప్రమాదం {proba:.1%}). "
            f"ముఖ్య సంకేతాలు: {why}. బ్యాంక్ విశ్లేషకుడు తనిఖీ చేసే వరకు నిధులు కదలవు."
        )
    return (
        f"HOLD: Payment blocked/delayed for review (scam risk {proba:.1%}). "
        f"Main signals: {why}. "
        f"Funds will not move until a bank analyst checks this."
    )


def hold_message_operator(row, proba, lang="en"):
    keys = hold_reason_keys(row)
    why = "; ".join(HOLD_REASON_TEXT[k][lang] for k in keys)
    try:
        dest = str(df_model.loc[row.name, "nameDest"])
        orig = str(df_model.loc[row.name, "nameOrig"])
    except Exception:
        dest, orig = "TOKEN_DEST", "TOKEN_ORIG"
    if lang == "hi":
        return (
            f"OPERATOR HOLD: केस गंभीर। स्कैम जोखिम {proba:.1%}। "
            f"प्रेषक={orig}, प्राप्तकर्ता={dest}। कारण: {why}। "
            f"कार्रवाई: राशि जारी न करें; धोखाधड़ी समीक्षा खोलें; ग्राहक से संपर्क करें।"
        )
    if lang == "te":
        return (
            f"OPERATOR HOLD: కేసు తీవ్రం. స్కామ్ ప్రమాదం {proba:.1%}. "
            f"పంపినవారు={orig}, స్వీకర్త={dest}. కారణాలు: {why}. "
            f"చర్య: నిధులు విడుదల చేయవద్దు; ఫ్రాడ్ రివ్యూ తెరవండి; కస్టమర్‌ను సంప్రదించండి."
        )
    return (
        f"OPERATOR HOLD: CRITICAL case. Scam risk {proba:.1%}. "
        f"Sender={orig}, Receiver={dest}. Reasons: {why}. "
        f"Action: Do not release funds; open fraud review; contact customer."
    )


def build_message(row, proba, decision, lang="en"):
    if decision == "ALLOW":
        return ""
    if decision == "HOLD":
        return hold_message_sender(row, proba, lang=lang)

    flags, txn_type = receiving_flags(row)
    keys = [k for k in flags if k in WARN_CATALOG] or ["generic_medium_risk"]
    try:
        dest_name = str(df_model.loc[row.name, "nameDest"])
    except Exception:
        dest_name = "TOKENIZED_DEST"

    header = {
        "en": f"WARN for SENDER (you decide): scam risk {proba:.1%}. Receiver: {dest_name}",
        "hi": f"चेतावनी (प्रेषक तय करें): स्कैम जोखिम {proba:.1%}। प्राप्तकर्ता: {dest_name}",
        "te": f"హెచ్చరిక (పంపేవారు నిర్ణయం): స్కామ్ ప్రమాదం {proba:.1%}. స్వీకర్త: {dest_name}",
    }[lang]

    lines = [header, "Why we warn:"]
    for k in keys:
        lines.append("- " + WARN_CATALOG[k][lang])
    return "\\n".join(lines)


print("=" * 72)
print("ALL POSSIBLE WARN TEMPLATES (EN / HI / TE) — for judges")
print("=" * 72)
for i, (k, tri) in enumerate(WARN_CATALOG.items(), 1):
    print(f"\\n[{i}] {k}")
    print(" EN:", tri["en"])
    print(" HI:", tri["hi"])
    print(" TE:", tri["te"])


ALL POSSIBLE WARN TEMPLATES (EN / HI / TE) — for judges
\n[1] receiver_zero_balance_history
 EN: The receiving account shows little/no balance history (possible new or emptied mule account). Confirm the recipient before paying.
 HI: प्राप्तकर्ता खाते में बैलेंस इतिहास कम/नहीं है (नया या खाली म्यूल अकाउंट हो सकता है)। भुगतान से पहले पुष्टि करें।
 TE: స్వీకర్త ఖాతాలో బ్యాలెన్స్ చరిత్ర తక్కువ/లేదు (కొత్త లేదా ఖాళీ మ్యూల్ ఖాతా కావచ్చు). చెల్లించే ముందు నిర్ధారించండి.
\n[2] receiver_had_zero_before_credit
 EN: Receiver balance was zero before this credit — a common scam pattern. Double-check who you are paying.
 HI: इस क्रेडिट से पहले प्राप्तकर्ता बैलेंस शून्य था — आम स्कैम पैटर्न। पुष्टि करें कि आप किसे भुगतान कर रहे हैं।
 TE: ఈ క్రెడిట్‌కు ముందు స్వీకర్త బ్యాలెన్స్ సున్నా — సాధారణ స్కామ్ నమూనా. ఎవరికి చెల్లిస్తున్నారో మళ్లీ తనిఖీ చేయండి.
\n[3] receiver_balance_inconsistency
 EN: Receiver balance change does not match this payment cleanly. Pause and verify the destination.
 HI: प्राप्तकर्त

In [8]:
# Live demos: HOLD (sender + operator, 3 langs) + WARN (3 langs)
print("HOLD examples — SENDER + BANK OPERATOR (EN/HI/TE):")
for idx in y_test[y_test == 1].index[:2]:
    r = X_test.loc[idx]
    p = float(model.predict_proba(pd.DataFrame([r]))[:, 1][0])
    d = decide(p)
    print(f"\n=== amount={r['amount']:.2f} proba={p:.4f} -> {d} ===")
    print("--- SENDER ---")
    for lang in ("en", "hi", "te"):
        print(f"[{lang.upper()}] {hold_message_sender(r, p, lang)}")
    print("--- BANK OPERATOR ---")
    for lang in ("en", "hi", "te"):
        print(f"[{lang.upper()}] {hold_message_operator(r, p, lang)}")

print("\n" + "=" * 72)
print("WARN examples (EN + HI + TE):")
warn_n = 0
for idx in X_test.index:
    r = X_test.loc[idx]
    p = float(model.predict_proba(pd.DataFrame([r]))[:, 1][0])
    d = decide(p)
    if d != "WARN":
        continue
    print(f"\n--- amount={r['amount']:.2f} proba={p:.4f} fraud_true={int(y_test.loc[idx])} ---")
    print(build_message(r, p, d, "en"))
    print(build_message(r, p, d, "hi"))
    print(build_message(r, p, d, "te"))
    warn_n += 1
    if warn_n >= 3:
        break
if warn_n == 0:
    print("Few/no WARN rows in scan — catalog above still shows all templates for judges.")


HOLD examples — SENDER + BANK OPERATOR (EN/HI/TE):

=== amount=20364.00 proba=1.0000 -> HOLD ===
--- SENDER ---
[EN] HOLD: Payment blocked/delayed for review (scam risk 100.0%). Main signals: weak/empty receiver balance history; receiver balance was zero before this credit. Funds will not move until a bank analyst checks this.
[HI] HOLD: भुगतान समीक्षा के लिए रोका/विलंबित (स्कैम जोखिम 100.0%)। मुख्य संकेत: प्राप्तकर्ता बैलेंस इतिहास कम/खाली; इस क्रेडिट से पहले प्राप्तकर्ता बैलेंस शून्य था। बैंक विश्लेषक जाँच तक राशि नहीं जाएगी।
[TE] HOLD: చెల్లింపు సమీక్ష కోసం ఆపబడింది/ఆలస్యం (స్కామ్ ప్రమాదం 100.0%). ముఖ్య సంకేతాలు: స్వీకర్త బ్యాలెన్స్ చరిత్ర బలహీనం/ఖాళీ; ఈ క్రెడిట్‌కు ముందు స్వీకర్త బ్యాలెన్స్ సున్నా. బ్యాంక్ విశ్లేషకుడు తనిఖీ చేసే వరకు నిధులు కదలవు.
--- BANK OPERATOR ---
[EN] OPERATOR HOLD: CRITICAL case. Scam risk 100.0%. Sender=C1715235345, Receiver=C1801554498. Reasons: weak/empty receiver balance history; receiver balance was zero before this credit. Action: Do not release funds;

## 8. Bank manager / analyst monitor queue
HOLD and WARN create investigation cases with evidence (operator review — not absolute legal proof).


In [9]:
def build_operator_case(row, proba, decision, fraud_true=None):
    if decision == "ALLOW":
        return None
    flags, txn_type = receiving_flags(row)
    try:
        dest_name = str(df_model.loc[row.name, "nameDest"])
        orig_name = str(df_model.loc[row.name, "nameOrig"])
    except Exception:
        dest_name, orig_name = "TOKEN_DEST", "TOKEN_ORIG"

    evidence = []
    if "high_fan_in_mule" in flags:
        evidence.append(f"High fan-in: {int(row.get('dest_fan_in', 0))} unique senders to receiver")
    if "receiver_zero_balance_history" in flags:
        evidence.append("Receiver zero-balance history")
    if "receiver_had_zero_before_credit" in flags:
        evidence.append("Receiver had zero balance before credit")
    if "receiver_balance_inconsistency" in flags:
        evidence.append(f"Dest balance error={float(row['errorBalanceDest']):.2f}")
    if "large_transfer_to_receiver" in flags:
        evidence.append(f"Large amount={float(row['amount']):.2f}")
    if any(k.startswith("risky_type_") for k in flags):
        evidence.append(f"Risky type={txn_type}")
    if not evidence:
        evidence.append("Elevated combined scam probability from model features")

    return {
        "case_id": f"SSZ-{row.name}",
        "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "queue": "BANK_MANAGER_MONITOR",
        "decision": decision,
        "severity": "CRITICAL-BLOCK" if decision == "HOLD" else "MEDIUM-REVIEW",
        "scam_probability": round(float(proba), 4),
        "amount": float(row["amount"]),
        "sender_account_token": orig_name,
        "receiver_account_token": dest_name,
        "txn_type": txn_type,
        "evidence_for_investigation": " | ".join(evidence),
        "operator_action": (
            "Do not release funds. Open fraud review."
            if decision == "HOLD"
            else "Verify payee / step-up auth before release. Monitor receiver."
        ),
        "operator_message_en": (
            hold_message_operator(row, proba, "en") if decision == "HOLD"
            else f"OPERATOR WARN: review recommended. Risk={proba:.1%}."
        ),
        "operator_message_hi": (
            hold_message_operator(row, proba, "hi") if decision == "HOLD"
            else f"OPERATOR WARN: समीक्षा करें। जोखिम={proba:.1%}।"
        ),
        "operator_message_te": (
            hold_message_operator(row, proba, "te") if decision == "HOLD"
            else f"OPERATOR WARN: సమీక్ష చేయండి. ప్రమాదం={proba:.1%}."
        ),
        "sender_message_en": build_message(row, proba, decision, "en"),
        "label_if_known": int(fraud_true) if fraud_true is not None else None,
    }


operator_queue = []
for idx in X_test.index[:8000]:
    r = X_test.loc[idx]
    p = float(model.predict_proba(pd.DataFrame([r]))[:, 1][0])
    d = decide(p)
    case = build_operator_case(r, p, d, fraud_true=y_test.loc[idx])
    if case:
        operator_queue.append(case)
    if len(operator_queue) >= 20:
        break

ops_df = pd.DataFrame(operator_queue)
print("BANK MANAGER / ANALYST QUEUE (sample)")
print(ops_df[[
    "case_id", "decision", "severity", "scam_probability",
    "amount", "receiver_account_token", "operator_action"
]].to_string(index=False))

print("\nFULL CASE FILE EXAMPLE:")
if len(operator_queue):
    for k, v in operator_queue[0].items():
        print(f"  {k}: {v}")

ops_df.to_csv("scamshield_operator_queue.csv", index=False)
print("\nSaved scamshield_operator_queue.csv")


BANK MANAGER / ANALYST QUEUE (sample)
    case_id decision       severity  scam_probability     amount receiver_account_token                                               operator_action
SSZ-1396139     HOLD CRITICAL-BLOCK            1.0000   20364.00            C1801554498                      Do not release funds. Open fraud review.
 SSZ-956971     HOLD CRITICAL-BLOCK            1.0000 2155569.91            C1486162037                      Do not release funds. Open fraud review.
 SSZ-910852     HOLD CRITICAL-BLOCK            1.0000   49434.25            C1814019284                      Do not release funds. Open fraud review.
SSZ-1726785     HOLD CRITICAL-BLOCK            1.0000  430722.59             C207357507                      Do not release funds. Open fraud review.
SSZ-1444632     HOLD CRITICAL-BLOCK            1.0000  305710.24             C757160228                      Do not release funds. Open fraud review.
 SSZ-626137     WARN  MEDIUM-REVIEW            0.4884  812443.

## 9. Latency benchmark (LinuxONE scoring window)
Abstract: measure latency per transaction / throughput.


In [10]:
bench_n = min(5000, len(X_test))
Xb = X_test.iloc[:bench_n]

t0 = time.time()
_ = model.predict_proba(Xb)
elapsed = time.time() - t0

per_txn_ms = (elapsed / bench_n) * 1000
tps = bench_n / elapsed if elapsed > 0 else float("inf")
print(f"Scored {bench_n} payments in {elapsed:.4f}s")
print(f"Latency ~ {per_txn_ms:.4f} ms / payment")
print(f"Throughput ~ {tps:.1f} payments / second")


Scored 5000 payments in 0.0073s
Latency ~ 0.0015 ms / payment
Throughput ~ 687929.1 payments / second


## 10. ONNX / model export
Matches abstract gradient-boosted → ONNX path. Falls back to XGBoost JSON if converters missing.


In [11]:
onnx_path = "scamshield_z.onnx"
exported = False

if HAS_ONNX:
    try:
        initial_type = [("input", FloatTensorType([None, X_train.shape[1]]))]
        onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)
        onnxmltools.utils.save_model(onnx_model, onnx_path)
        exported = True
        print("Exported", onnx_path)
    except Exception as e:
        print("onnxmltools path failed:", e)

if not exported:
    try:
        model.save_model("scamshield_z.json")
        print("Saved native XGBoost JSON: scamshield_z.json")
        print("ONNX converter not available in this image; JSON model still portable for demo.")
    except Exception as e:
        print("Model save failed:", e)


Saved native XGBoost JSON: scamshield_z.json
ONNX converter not available in this image; JSON model still portable for demo.


## 11. Optional — Credit Card Fraud (ULB) imbalance sanity check
From abstract. Runs only if `creditcard.csv` is uploaded.


In [12]:
cc_paths = [Path("creditcard.csv"), Path("shared/creditcard.csv")]
cc_path = next((p for p in cc_paths if p.exists()), None)

if cc_path is None:
    print("creditcard.csv not found — skip (optional). Upload from Kaggle ULB if you want this check.")
else:
    cc = pd.read_csv(cc_path)
    print("ULB creditcard shape:", cc.shape, "fraud:", int(cc["Class"].sum()))
    feat_cols = [c for c in cc.columns if c not in ("Class", "Time")]
    Xc = cc[feat_cols]
    yc = cc["Class"]
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
        Xc, yc, test_size=0.2, random_state=42, stratify=yc
    )
    spw = (yc_tr == 0).sum() / max((yc_tr == 1).sum(), 1)
    cc_model = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=spw, eval_metric="aucpr",
        random_state=42, n_jobs=-1, early_stopping_rounds=20,
    )
    Xc_tr2, Xc_val, yc_tr2, yc_val = train_test_split(
        Xc_tr, yc_tr, test_size=0.2, random_state=42, stratify=yc_tr
    )
    cc_model.fit(Xc_tr2, yc_tr2, eval_set=[(Xc_val, yc_val)], verbose=False)
    pp = cc_model.predict_proba(Xc_te)[:, 1]
    print("ULB TEST ROC-AUC:", round(roc_auc_score(yc_te, pp), 4))
    print("ULB TEST PR-AUC :", round(average_precision_score(yc_te, pp), 4))


creditcard.csv not found — skip (optional). Upload from Kaggle ULB if you want this check.


## 12. Summary for judges
- PaySim 2M on IBM LinuxONE
- Behaviour + mule fan-in features
- XGBoost with val early-stopping; PR-AUC for imbalance
- ALLOW (silent) / WARN (EN·HI·TE) / HOLD (block + analyst)
- Bank-manager evidence queue CSV
- Latency/TPS benchmark + model export


In [13]:
print("ScamShield Z ready on IBM LinuxONE")
print("Decisions: ALLOW | WARN (multilingual) | HOLD -> analyst queue")
print("Artifacts: scamshield_operator_queue.csv, model export if available")


ScamShield Z ready on IBM LinuxONE
Decisions: ALLOW | WARN (multilingual) | HOLD -> analyst queue
Artifacts: scamshield_operator_queue.csv, model export if available
